In [2]:
pip install geopy


   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

def elabora_gtfs_senza_file(gtfs_dir=".", output_excel="riepilogo_gtfs_online.xlsx"):
    print("🚀 [1/5] Caricamento dei file GTFS...")
    routes = pd.read_csv(os.path.join(gtfs_dir, 'routes.txt'))
    trips = pd.read_csv(os.path.join(gtfs_dir, 'trips.txt'))
    stop_times = pd.read_csv(os.path.join(gtfs_dir, 'stop_times.txt'))
    stops = pd.read_csv(os.path.join(gtfs_dir, 'stops.txt'))
    
    calendar_path = os.path.join(gtfs_dir, 'calendar.txt')
    has_calendar = os.path.exists(calendar_path)
    if has_calendar:
        calendar = pd.read_csv(calendar_path)

    print("⏳ [2/5] Ordinamento delle fermate e calcolo dei capolinea...")
    stop_times['stop_sequence'] = pd.to_numeric(stop_times['stop_sequence'])
    stop_times = stop_times.sort_values(['trip_id', 'stop_sequence'])
    
    first_stops = stop_times.groupby('trip_id').first().reset_index()
    first_stops = first_stops[['trip_id', 'stop_id', 'departure_time']]
    first_stops.rename(columns={'stop_id': 'start_stop_id', 'departure_time': 'departure_time'}, inplace=True)
    
    last_stops = stop_times.groupby('trip_id').last().reset_index()
    last_stops = last_stops[['trip_id', 'stop_id', 'arrival_time']]
    last_stops.rename(columns={'stop_id': 'end_stop_id', 'arrival_time': 'arrival_time'}, inplace=True)
    
    trip_endpoints = pd.merge(first_stops, last_stops, on='trip_id')

    print("⏳ [3/5] Unione delle anagrafiche (Fermate e Linee)...")
    trip_endpoints = pd.merge(trip_endpoints, stops[['stop_id', 'stop_name']], left_on='start_stop_id', right_on='stop_id', how='left')
    trip_endpoints.rename(columns={'stop_name': 'fermata_partenza'}, inplace=True)
    trip_endpoints.drop(columns=['stop_id', 'start_stop_id'], inplace=True)
    
    trip_endpoints = pd.merge(trip_endpoints, stops[['stop_id', 'stop_name']], left_on='end_stop_id', right_on='stop_id', how='left')
    trip_endpoints.rename(columns={'stop_name': 'fermata_arrivo'}, inplace=True)
    trip_endpoints.drop(columns=['stop_id', 'end_stop_id'], inplace=True)
    
    trip_full = pd.merge(trip_endpoints, trips, on='trip_id', how='left')
    
    if 'route_short_name' in routes.columns:
        routes['linea'] = routes['route_short_name'].fillna(routes['route_long_name'])
    else:
        routes['linea'] = routes['route_long_name']
        
    trip_full = pd.merge(trip_full, routes[['route_id', 'linea']], on='route_id', how='left')

    print("⏳ [4/5] Calcolo dei giorni totali di validità nel periodo...")
    if has_calendar:
        days = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
        
        def calculate_total_days(row):
            try:
                start = datetime.strptime(str(int(row['start_date'])), '%Y%m%d')
                end = datetime.strptime(str(int(row['end_date'])), '%Y%m%d')
                total_days = 0
                curr = start
                while curr <= end:
                    if row.get(days[curr.weekday()]) == 1:
                        total_days += 1
                    curr += timedelta(days=1)
            except Exception:
                total_days = 0
            return pd.Series([total_days, row['start_date'], row['end_date']])

        calendar[['giorni_validita', 'start_date', 'end_date']] = calendar.apply(calculate_total_days, axis=1)
        trip_full = pd.merge(trip_full, calendar[['service_id', 'giorni_validita', 'start_date', 'end_date']], on='service_id', how='left')
    else:
        trip_full['giorni_validita'] = 0

    output_columns = [
        'linea', 'trip_id', 'trip_headsign', 
        'fermata_partenza', 'departure_time', 
        'fermata_arrivo', 'arrival_time', 
        'giorni_validita', 'start_date', 'end_date'
    ]
    existing_cols = [col for col in output_columns if col in trip_full.columns]
    result_df = trip_full[existing_cols]

    print("⏳ [5/5] Geolocalizzazione online delle fermate (potrebbe richiedere qualche secondo)...")
    stops_clean = stops[['stop_id', 'stop_name']].copy()
    
    geolocator = Nominatim(user_agent="gtfs_processor_script_python")
    # RateLimiter corretto (senza il parametro non supportato)
    geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1, swallow_exceptions=True)

    comuni = []
    province = []

    for idx, row in stops.iterrows():
        lat = row['stop_lat']
        lon = row['stop_lon']
        
        comune = "Non disponibile"
        provincia = "Non disponibile"
        
        try:
            location = geocode(f"{lat}, {lon}")
            if location and location.raw and 'address' in location.raw:
                address = location.raw['address']
                comune = address.get('city', address.get('town', address.get('village', address.get('municipality', 'Non disponibile'))))
                provincia = address.get('county', address.get('state_district', 'Non disponibile'))
        except Exception:
            pass
            
        comuni.append(comune)
        province.append(provincia)

    stops_clean['comune'] = comuni
    stops_clean['provincia'] = province

    print("💾 Salvataggio del file Excel definitivo...")
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        result_df.to_excel(writer, sheet_name='Riepilogo Corse', index=False)
        stops_clean.to_excel(writer, sheet_name='Fermate', index=False)
        
    print(f"✅ Fatto! File Excel salvato con successo come: {output_excel}")
    return result_df, stops_clean

# Esecuzione
df_corse, df_ferm = elabora_gtfs_senza_file()

🚀 [1/5] Caricamento dei file GTFS...
⏳ [2/5] Ordinamento delle fermate e calcolo dei capolinea...
⏳ [3/5] Unione delle anagrafiche (Fermate e Linee)...
⏳ [4/5] Calcolo dei giorni totali di validità nel periodo...
⏳ [5/5] Geolocalizzazione online delle fermate (potrebbe richiedere qualche secondo)...


KeyboardInterrupt: 